# Sports Player Tracking Pipeline (Google Colab Notebook)

End-to-end notebook for tracking players in sports videos:

1. **Setup** — install dependencies and configure paths
2. **YOLO v11** — player detection
3. **OpenPose** — body keypoint estimation
4. **ByteTrack** — multi-frame player tracking
5. **Full pipeline** — process all videos and save results

---
## Part 1: Setup & Install Dependencies

In [ ]:
# Check GPU availability (recommended for YOLO + OpenPose)
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install dependencies
!pip install -q ultralytics opencv-python controlnet-aux PyYAML tqdm matplotlib lap

In [ ]:
# Mount Google Drive 
from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/DS5216-PA2'  
%cd {PROJECT_ROOT}

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('.').resolve()))

from src.config_loader import load_config, get_video_paths

config = load_config()

# Update paths for Colab
config['paths']['video_dir'] = '/content/drive/MyDrive/SportsVideos'  
config['paths']['output_dir'] = '/content/drive/MyDrive/outputs'       

videos = get_video_paths(config)
print(f"Found {len(videos)} video(s):")
for v in videos:
    print(f"  - {v.name}")

In [ ]:
# Load first frame (reused in Parts 2 and 3)
if not videos:
    raise FileNotFoundError("No videos found. Check video_dir path in config.")

cap = cv2.VideoCapture(str(videos[0]))
ret, frame = cap.read()
cap.release()

if ret:
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title(videos[0].name)
    plt.axis('off')
    plt.show()
    print(f"Frame shape: {frame.shape}")
else:
    raise RuntimeError(f"Could not read frame from {videos[0]}")

---
## Part 2: Player Detection with YOLO v11

In [ ]:
from src.detection.yolo_detector import YOLOPlayerDetector
from src.utils.visualization import draw_bounding_box

yolo_cfg = config['models']['yolo']
detector = YOLOPlayerDetector(
    model_name=yolo_cfg['model_name'],
    confidence=yolo_cfg['confidence'],
    iou=yolo_cfg['iou'],
    person_class_id=yolo_cfg['person_class_id'],
)

detections = detector.detect(frame)
print(f"Detected {len(detections)} player(s) in {videos[0].name}")
for i, det in enumerate(detections):
    print(f"  Player {i + 1}: bbox={det.bbox}, confidence={det.confidence:.2f}")

In [ ]:
vis = frame.copy()
for det in detections:
    vis = draw_bounding_box(
        vis, det.bbox, f"Player ({det.confidence:.2f})",
        color=(0, 255, 0), thickness=2,
    )

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f"YOLO v11 Detection - {len(detections)} players")
plt.axis('off')
plt.show()

---
## Part 3: Keypoint Detection with OpenPose

In [ ]:
from src.keypoints.openpose_estimator import OpenPoseEstimator, BODY_JOINT_NAMES
from src.utils.visualization import draw_keypoints

if not detections:
    raise ValueError("No players detected. Try lowering confidence threshold in config.yaml.")

bbox = detections[0].bbox
print(f"Using player bbox: {bbox}")

openpose_cfg = config['models']['openpose']
pose_estimator = OpenPoseEstimator(
    device=openpose_cfg['device'],
    include_hands=openpose_cfg['include_hands'],
    include_face=openpose_cfg['include_face'],
)

keypoints = pose_estimator.estimate_on_crop(frame, bbox)

if keypoints is not None:
    visible = sum(1 for kp in keypoints if kp[2] > 0.1)
    print(f"Detected {visible} visible keypoints out of 18")
    for name, (x, y, conf) in zip(BODY_JOINT_NAMES, keypoints):
        if conf > 0.1:
            print(f"  {name}: ({x:.0f}, {y:.0f}) conf={conf:.2f}")
else:
    print("No pose detected in crop.")

In [ ]:
vis = frame.copy()
if keypoints is not None:
    vis = draw_keypoints(vis, keypoints, (0, 0, 255), (255, 128, 0))

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title("OpenPose Keypoints on Player")
plt.axis('off')
plt.show()

---
## Part 4: Player Tracking with ByteTrack

In [ ]:
from src.tracking.player_tracker import PlayerTracker

track_cfg = config['models']['tracking']
tracker = PlayerTracker(
    model_name=yolo_cfg['model_name'],
    confidence=yolo_cfg['confidence'],
    tracker=track_cfg['tracker'],
    persist=track_cfg['persist'],
)

NUM_FRAMES = 30
cap = cv2.VideoCapture(str(videos[0]))
track_history = {}

for frame_idx in range(NUM_FRAMES):
    ret, track_frame = cap.read()
    if not ret:
        break

    tracked = tracker.track(track_frame)
    for player in tracked:
        track_history.setdefault(player.track_id, []).append(
            (frame_idx, player.bbox)
        )

cap.release()

print(f"Tracked {len(track_history)} unique player(s) over {NUM_FRAMES} frames:")
for tid, history in track_history.items():
    print(f"  Player ID {tid}: seen in {len(history)} frames")

In [ ]:
# Visualize tracking on frame 15
cap = cv2.VideoCapture(str(videos[0]))
for _ in range(15):
    cap.read()
ret, track_frame = cap.read()
cap.release()

tracked = tracker.track(track_frame)
vis = track_frame.copy()
colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255)]

for player in tracked:
    color = colors[player.track_id % len(colors)]
    vis = draw_bounding_box(
        vis, player.bbox, f"ID {player.track_id}", color, thickness=2
    )

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f"ByteTrack - {len(tracked)} players tracked")
plt.axis('off')
plt.show()

---
## Part 5: Full Pipeline — Track All Sports Videos

In [ ]:
from src.pipeline.tracking_pipeline import PlayerTrackingPipeline

# Quick test: 50 frames per video (set to None for full video)
config['processing']['max_frames'] = 50
config['processing']['save_video'] = True
config['processing']['save_json'] = True

print(f"Will process {len(videos)} video(s):")
for v in videos:
    print(f"  - {v.name}")

In [ ]:
pipeline = PlayerTrackingPipeline(config)
results = pipeline.process_all_videos(videos)

In [ ]:
for result in results:
    video_name = Path(result['video']).name
    num_frames = len(result['frames'])
    total_players = sum(len(f['players']) for f in result['frames'])
    print(f"{video_name}: {num_frames} frames, {total_players} total player detections")

In [ ]:
from IPython.display import Video, display

output_dir = Path(config['paths']['output_dir']) / videos[0].stem
output_video = output_dir / f"{videos[0].stem}_tracked.mp4"

if output_video.exists():
    display(Video(str(output_video), width=640))
else:
    print(f"Output not found: {output_video}")

In [ ]:
# Process ALL frames on ALL videos 
config['processing']['max_frames'] = None
pipeline = PlayerTrackingPipeline(config)
results = pipeline.process_all_videos(videos)